# 05 · T-DEED-Evidenz mit Spatial Hybrid v1 verbinden

Die Featuretabelle aus Notebook 04 ergänzt die bestehende v1-Vorhersage in zwei Varianten: A bestätigt Kontakte, B kann bestehende H/A-Grenzen begrenzt verschieben. Die fünf Pilotclips bleiben der gesamte Interventionsbereich. Ein besserer öffentlicher Score wird hier nicht gesucht.

[← Notebook 04](../04_kontaktdiagnose/04_kontaktdiagnose.ipynb)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from IPython.display import display

## 1. Dateien und feste Parameter

Video-Support erhält einen Dreieckskern von ±0,5 s. Variante B darf eine bestehende Grenze um höchstens 0,4 s zusätzlich verschieben; Nachbarphasen begrenzen den Radius weiter. DRIVE dient nur als Kontinuitätssignal.

In [ ]:
PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents)
               if (p / "artifacts/spatial_hybrid_v1/references/spatial_hybrid_v1.csv").is_file())
ALIGN = PROJECT / "output/video_event_spotting_pilot/integration_diagnostics"
PILOT = PROJECT / "output/video_event_spotting_pilot"
OUT = PROJECT / "output/spatial_hybrid_v2_pilot"
for part in ("diagnostics", "figures", "submissions"):
    (OUT / part).mkdir(parents=True, exist_ok=True)
DIA, FIG, SUB = OUT / "diagnostics", OUT / "figures", OUT / "submissions"
FPS, N_FRAMES, RANDOM_STATE = 25, 199319, 42
PARAMS = {"seed": RANDOM_STATE, "support_halfwidth_s": 0.5, "contact_matching_window_s": 0.5,
          "max_additional_shift_s": 0.4, "native_stride_frames": 2, "contact_crossing_tolerance_frames": 2,
          "boundary_action_classes": ["PASS", "HIGH PASS"], "short_phase_diagnostic_seconds": 4.0,
          "aggregation": "maximum positive support", "tie_rule": "keep original boundary",
          "team_source": "existing tracking consensus, unvalidated", "A": "positive contact confirmation; no unsupported hard-label change",
          "B": "spatial crossing + contact + video, bounded additional alignment"}

## 2. Featuretabelle und Modellabdeckung laden

Nur vollständig abgedeckte LIVE-H/A-Frames innerhalb der Clips kommen für Änderungen infrage. Fehlende Abdeckung liefert keine negative Ereignisevidenz.

In [ ]:
features = pd.read_csv(ALIGN / "tdeed_tracking_event_features.csv")

contacts = pd.read_csv(ALIGN / "tables/original_tracking_contacts.csv")

pairs = pd.read_csv(ALIGN / "tables/one_to_one_event_matches.csv")

spatial = pd.read_csv(ALIGN / "tables/spatial_live_crossing_reference_events.csv")

clips = pd.read_csv(PILOT / "diagnostics/clip_manifest.csv")

v1_path = PROJECT / "artifacts/spatial_hybrid_v1/references/spatial_hybrid_v1.csv"

v1_table = pd.read_csv(v1_path)

v1 = v1_table.label.to_numpy(dtype="<U1")

with np.load(PROJECT / "artifacts/spatial_hybrid_v1/inputs/spatial_hybrid_inputs_v1.npz", allow_pickle=False) as blob:
    status, halves = blob["status"].copy(), blob["halves"].copy()

clip_mask, model_coverage = np.zeros(N_FRAMES, bool), np.zeros(N_FRAMES, bool)

clip_owner = np.full(N_FRAMES, -1, int)

for number, scene in enumerate(clips.itertuples()):
    indices = np.arange(scene.first_frame, scene.stop_frame_exclusive)
    clip_mask[indices], clip_owner[indices] = True, number
    with np.load(PILOT / f"diagnostics/model_scores/{scene.scene_id}.npz", allow_pickle=False) as blob:
        dense_coverage = np.repeat(blob["window_coverage"] > 0, PARAMS["native_stride_frames"])
        model_coverage[indices] = dense_coverage[:len(indices)]

active = np.zeros(N_FRAMES, bool)

for first, last in halves:
    active[first:last + 1] = True

live = active & (status == "L")

intervention_mask = model_coverage & live & np.isin(v1, ["H", "A"])

print("Clipframes:", int(clip_mask.sum()), "| tatsächlich modellabgedeckt:", int(model_coverage.sum()), "| LIVE-Interventionsmaske:", int(intervention_mask.sum()))

display(features.groupby(["video_event_type", "team_attribution_status"]).size().unstack(fill_value=0))

## 3. Positive Videoevidenz auf dem Frameraster

Der unkalibrierte Score wird lokal mit dem Dreieckskern gewichtet. Bei Überlappung zählt der maximale positive Support. Trackingkonsens bleibt ein eigener Kanal.

In [ ]:
def kernel(distance_frames):
    return np.maximum(0, 1 - np.abs(distance_frames) / (PARAMS["support_halfwidth_s"] * FPS))


channels = {name: np.zeros(N_FRAMES, np.float32) for name in ["PASS", "DRIVE", "HIGH PASS", "tracking_H", "tracking_A"]}
for event in features.itertuples():
    scene = clips.loc[clips.scene_id.eq(event.scene_id)].iloc[0]
    radius = int(np.ceil(PARAMS["support_halfwidth_s"] * FPS))
    frames = np.arange(max(int(scene.first_frame), event.frame - radius), min(int(scene.stop_frame_exclusive), event.frame + radius + 1))
    frames = frames[model_coverage[frames]]
    values = event.video_event_score * kernel(frames - event.frame)
    channels[event.video_event_type][frames] = np.maximum(channels[event.video_event_type][frames], values)
    if event.team_attribution_status == "tracking_consensus":
        name = "tracking_" + event.tracking_attributed_team
        channels[name][frames] = np.maximum(channels[name][frames], values)
np.savez_compressed(DIA / "video_evidence_full_framegrid.npz", **channels, clip_mask=clip_mask,
                    model_coverage=model_coverage, intervention_mask=intervention_mask, fps=FPS)
dense_table = pd.DataFrame({"frame": np.flatnonzero(clip_mask), **{name: values[clip_mask] for name, values in channels.items()},
                           "model_coverage": model_coverage[clip_mask], "intervention_mask": intervention_mask[clip_mask]})
dense_table.to_csv(DIA / "video_evidence_pilot_framegrid.csv", index=False)
print("47 unveränderte Eventpunkte lokalisiert; keine neue Videoinferenz und keine negative Evidenz aus fehlenden Events.")

## 4. Variante A: Kontakte bestätigen

Zeitlich eindeutige Kontakte im selben abgedeckten Clip erhalten zusätzlichen Support. Da das Modell weder handelndes Team noch kontrollierte Übernahme unabhängig erkennt, verändert diese Variante keine harten Besitzlabels.

In [ ]:
def confirm_contacts(events, original_contacts, primary_pairs, base, coverage, owner):
    records = []
    contact_by_frame = original_contacts.set_index("frame")
    matches = primary_pairs.set_index("event_id")
    for event in events.itertuples():
        pair = matches.loc[event.event_id]
        reason, support, contact_frame, contact_team = "no_contact_match", 0.0, None, None
        if pair.matched:
            contact_frame = int(pair.reference_frame)
            contact_team = str(contact_by_frame.loc[contact_frame, "team"])
            if pair.ambiguous:
                reason = "ambiguous_temporal_contact_match"
            elif not coverage[contact_frame] or owner[contact_frame] != owner[event.frame]:
                reason = "contact_outside_same_model_covered_clip"
            else:
                support = float(event.video_event_score * kernel(contact_frame - event.frame))
                reason = "positive_continuity_support" if event.video_event_type == "DRIVE" else "positive_ball_action_support"
        records.append({"event_id": event.event_id, "event_frame": int(event.frame), "event_type": event.video_event_type,
                        "contact_frame": contact_frame, "contact_team_from_tracking": contact_team,
                        "positive_contact_support": support, "accepted_confirmation": support > 0,
                        "reason": reason, "existing_label_at_contact": str(base[contact_frame]) if contact_frame is not None else None,
                        "tracking_consensus": event.team_attribution_status == "tracking_consensus",
                        "independent_video_team": False, "label_change_authorized_by_A": False})
    return base.copy(), pd.DataFrame(records)


primary_pairs = pairs.loc[pairs.source.eq("tracking_contact") & pairs.window_s.eq(PARAMS["contact_matching_window_s"])]
v2_A, confirmation_A = confirm_contacts(features, contacts, primary_pairs, v1, model_coverage, clip_owner)
confirmation_A.to_csv(DIA / "variant_A_contact_confirmations.csv", index=False)
display(confirmation_A.loc[confirmation_A.accepted_confirmation])
print("Zusätzlich gestützte Kontakte:", int(confirmation_A.accepted_confirmation.sum()), "| A ändert keine unbelegten H/A-Labels.")

## 5. Variante B: Bestehende Grenzen ausrichten

Ein PASS-/HIGH-PASS-Anker muss über Kontakt und Trackingkonsens zum neuen Team passen. Eine Grenze wird nur zu einem vorhandenen räumlichen Crossing verschoben, wenn der positive gemeinsame Support steigt und alle geänderten Frames im abgedeckten LIVE-Bereich liegen. Bei Gleichstand bleibt die Grenze erhalten.

In [ ]:
def make_boundary_anchors(events, original_contacts, coverage, owner):
    contact_by_frame = original_contacts.set_index("frame")
    rows = []
    for event in events.itertuples():
        eligible, reason = False, "eligible_boundary_action_anchor"
        contact_frame = int(event.one_to_one_contact_frame) if pd.notna(event.one_to_one_contact_frame) else None
        if event.video_event_type not in PARAMS["boundary_action_classes"]:
            reason = "DRIVE_is_continuity_only"
        elif event.team_attribution_status != "tracking_consensus":
            reason = "no_unambiguous_automatic_tracking_team"
        elif contact_frame is None or event.contact_matching_ambiguous_0_5s:
            reason = "no_unambiguous_contact_assignment"
        elif not coverage[contact_frame] or owner[contact_frame] != owner[event.frame]:
            reason = "contact_outside_same_model_covered_clip"
        elif contact_by_frame.loc[contact_frame, "team"] != event.tracking_attributed_team:
            reason = "contact_team_disagrees_with_tracking_consensus"
        else:
            eligible = True
        rows.append({"event_id": event.event_id, "event_frame": int(event.frame), "event_type": event.video_event_type,
                     "score": float(event.video_event_score), "contact_frame": contact_frame,
                     "team": event.tracking_attributed_team if eligible else None, "eligible": eligible, "reason": reason})
    return pd.DataFrame(rows)


def live_segments(live_mask):
    edges = np.flatnonzero(np.diff(np.r_[False, live_mask, False].astype(np.int8)))
    return list(zip(edges[::2], edges[1::2]))


def align_boundaries(base, live_mask, coverage, owner, crossings, anchors):
    result = base.copy()
    boundary_rows, candidate_rows = [], []
    cap = round(PARAMS["max_additional_shift_s"] * FPS)
    for segment_start, segment_stop in live_segments(live_mask):
        frames = np.arange(segment_start + 1, segment_stop)
        changes = frames[np.isin(base[frames - 1], ["H", "A"]) & np.isin(base[frames], ["H", "A"]) & (base[frames - 1] != base[frames])]
        for number, boundary in enumerate(changes):
            old_team, new_team = base[boundary - 1], base[boundary]
            left = changes[number - 1] if number else segment_start
            right = changes[number + 1] if number + 1 < len(changes) else segment_stop
            radius = int(min(cap, (boundary - left - 1) // 2, (right - boundary - 1) // 2))
            available = bool(coverage[boundary])
            local_anchors = anchors.loc[anchors.eligible & anchors.team.eq(new_team)
                & anchors.event_frame.between(segment_start, segment_stop - 1)
                & anchors.contact_frame.between(segment_start, segment_stop - 1)]
            if available:
                local_anchors = local_anchors.loc[local_anchors.event_frame.map(lambda f: owner[int(f)] == owner[boundary])]
            else:
                local_anchors = local_anchors.iloc[:0]

            def joint_support(frame):
                if not coverage[frame] or owner[frame] != owner[boundary]:
                    return 0.0, None
                close = local_anchors.loc[(local_anchors.contact_frame - frame).abs() <= PARAMS["contact_crossing_tolerance_frames"]]
                if close.empty:
                    return 0.0, None
                values = close.score.to_numpy() * kernel(close.event_frame.to_numpy() - frame)
                best = int(np.argmax(values))
                return float(values[best]), str(close.iloc[best].event_id) if values[best] > 0 else None

            original_support, _ = joint_support(int(boundary)) if available else (0.0, None)
            candidates = crossings.loc[crossings.team.eq(new_team) & crossings.frame.between(boundary - radius, boundary + radius)] if available and radius > 0 else crossings.iloc[:0]
            proposed = []
            for crossing in candidates.itertuples():
                candidate = int(crossing.frame)
                if candidate == boundary:
                    continue
                span = np.arange(min(candidate, boundary), max(candidate, boundary))
                span_valid = bool(len(span) and coverage[span].all() and live_mask[span].all() and (owner[span] == owner[boundary]).all())
                support, event_id = joint_support(candidate) if span_valid else (0.0, None)
                better = support > original_support + 1e-12
                candidate_rows.append({"original_frame": int(boundary), "candidate_frame": candidate,
                    "from_team": str(old_team), "to_team": str(new_team), "shift_frames": candidate - int(boundary),
                    "span_inside_intervention_domain": span_valid, "positive_joint_support": support,
                    "original_joint_support": original_support, "supporting_event_id": event_id,
                    "eligible_improvement": bool(span_valid and better)})
                if span_valid and better:
                    proposed.append((support, abs(candidate - boundary), candidate, event_id))
            if proposed:
                best = sorted(proposed, key=lambda item: (-item[0], item[1], item[2]))[0]
                new_boundary, used_event, score = int(best[2]), best[3], float(best[0])
                if new_boundary < boundary:
                    result[new_boundary:boundary] = new_team
                else:
                    result[boundary:new_boundary] = old_team
                reason = "positive_video_contact_spatial_support_improved"
            else:
                new_boundary, used_event, score = int(boundary), None, original_support
                reason = ("outside_model_covered_clips" if not available else "no_phase_guard_room" if radius <= 0 else
                          "no_eligible_new_team_action_anchor_in_live_segment" if local_anchors.empty else
                          "no_nearby_spatial_crossing" if candidates.empty else "no_strict_joint_support_improvement")
            boundary_rows.append({"original_frame": int(boundary), "new_frame": new_boundary,
                "from_team": str(old_team), "to_team": str(new_team), "segment_start": int(segment_start), "segment_stop": int(segment_stop),
                "inside_model_coverage": available, "guarded_radius_frames": radius,
                "eligible_new_team_anchors_in_segment": len(local_anchors), "spatial_candidates": len(candidates),
                "shift_frames": new_boundary - int(boundary), "shift_seconds": (new_boundary - boundary) / FPS,
                "used_event_id": used_event, "joint_support": score, "reason": reason})
    return result, pd.DataFrame(boundary_rows), pd.DataFrame(candidate_rows, columns=["original_frame", "candidate_frame", "from_team", "to_team", "shift_frames", "span_inside_intervention_domain", "positive_joint_support", "original_joint_support", "supporting_event_id", "eligible_improvement"])


anchors_B = make_boundary_anchors(features, contacts, model_coverage, clip_owner)
v2_B, boundaries_B, candidates_B = align_boundaries(v1, live, model_coverage, clip_owner, spatial, anchors_B)
anchors_B.to_csv(DIA / "variant_B_event_anchor_eligibility.csv", index=False)
boundaries_B.to_csv(DIA / "variant_B_boundary_decisions.csv", index=False)
candidates_B.to_csv(DIA / "variant_B_spatial_candidate_decisions.csv", index=False)
display(boundaries_B.loc[boundaries_B.inside_model_coverage])
print("Zulässige automatische Boundary-Anker:", int(anchors_B.eligible.sum()), "| tatsächlich verschobene Grenzen:", int(boundaries_B.shift_frames.ne(0).sum()))

## 6. Änderungen vergleichen und Vorhersagen speichern

Die Tabelle zeigt geänderte Frames, Zeitversätze und kurze Besitzphasen gegenüber v1. Event- und Kandidatentabellen dokumentieren, welche Hinweise tatsächlich verwendet wurden.

In [ ]:
def phase_table(labels):
    starts = np.flatnonzero(np.r_[True, labels[1:] != labels[:-1]])
    stops = np.r_[starts[1:], len(labels)]
    return pd.DataFrame({"phase_id": np.arange(len(starts)), "start_frame": starts, "stop_frame_exclusive": stops,
                         "label": labels[starts], "frames": stops - starts, "duration_s": (stops - starts) / FPS})


def live_switch_frames(labels):
    return np.flatnonzero(live[:-1] & live[1:] & np.isin(labels[:-1], ["H", "A"]) & np.isin(labels[1:], ["H", "A"]) & (labels[:-1] != labels[1:])) + 1


models = {"spatial_hybrid_v1_reference": v1, "spatial_hybrid_v2_A": v2_A, "spatial_hybrid_v2_B": v2_B}
base_phases, base_switches = phase_table(v1), live_switch_frames(v1)
metrics, short_rows = [], []
for name, labels in models.items():
    changed = labels != v1
    phases, switches = phase_table(labels), live_switch_frames(labels)
    fixed_mask = np.isin(v1, ["D", "X"])
    metrics.append({"model": name, "changed_frames": int(changed.sum()), "changed_frames_in_clips": int((changed & clip_mask).sum()),
        "changed_fraction_inside_clips": float((changed & clip_mask).sum() / changed.sum()) if changed.any() else np.nan,
        "changed_frames_outside_clips": int((changed & ~clip_mask).sum()), "D_X_changed_frames": int((changed & fixed_mask).sum()),
        "HA_switches": len(switches), "additional_HA_switches": 0, "removed_HA_switches": 0,
        "shifted_HA_switches": int((switches != base_switches).sum()), "max_additional_shift_s": float(np.abs(switches - base_switches).max() / FPS) if len(switches) else 0,
        "short_HA_phases": int((phases.label.isin(["H", "A"]) & phases.duration_s.lt(4)).sum()),
        "short_phase_lengths_changed": int(((phases.frames != base_phases.frames) & phases.label.isin(["H", "A"]) & (phases.duration_s.lt(4) | base_phases.duration_s.lt(4))).sum())})
    short = phases.loc[phases.label.isin(["H", "A"]) & (phases.duration_s.lt(4) | base_phases.duration_s.lt(4))].copy()
    short["model"], short["original_frames"] = name, base_phases.loc[short.index, "frames"]
    short_rows.append(short)
    phases.to_csv(DIA / f"{name}_phase_table.csv", index=False)
    pd.DataFrame({"frame": np.arange(N_FRAMES), "label": labels}).to_csv(SUB / f"{name}.csv", index=False, lineterminator="\n")
comparison = pd.DataFrame(metrics)
comparison.to_csv(DIA / "variant_comparison.csv", index=False)
pd.concat(short_rows, ignore_index=True).to_csv(DIA / "short_possession_phase_comparison.csv", index=False)
used_B = set(boundaries_B.used_event_id.dropna())
event_use = features[["event_id", "scene_id", "frame", "video_event_type", "video_event_score", "team_attribution_status", "tracking_attributed_team"]].copy()
event_use = event_use.merge(confirmation_A[["event_id", "accepted_confirmation", "reason"]].rename(columns={"reason": "A_reason", "accepted_confirmation": "A_positive_contact_confirmation"}), on="event_id", validate="one_to_one")
event_use = event_use.merge(anchors_B[["event_id", "eligible", "reason"]].rename(columns={"eligible": "B_eligible_anchor", "reason": "B_anchor_reason"}), on="event_id", validate="one_to_one")
event_use["dense_evidence_channel"] = True
event_use["B_used_for_actual_boundary_shift"] = event_use.event_id.isin(used_B)
event_use.to_csv(DIA / "all_47_event_usage.csv", index=False)
np.savez_compressed(DIA / "all_full_match_prediction_arrays.npz", **models)
display(comparison)

## 7. Zeitlicher Vorher-/Nachher-Vergleich

Besitzfolgen, Kontakte und Videoevents werden pro Clip gegenübergestellt. Die Supportkurven zeigen, wo zusätzliche Evidenz vorliegt.

In [ ]:
colors = {"H": "#2474b5", "A": "#d55050", "D": "#999999", "X": "#ececec"}
label_codes = {label: i for i, label in enumerate(colors)}
event_colors = {"PASS": "#2a9d65", "DRIVE": "#c18421", "HIGH PASS": "#9453ad"}
fig, axes = plt.subplots(5, 1, figsize=(14, 14), constrained_layout=True)
for ax, scene in zip(axes, clips.itertuples()):
    start, stop = int(scene.first_frame), int(scene.stop_frame_exclusive)
    matrix = np.array([[label_codes[value] for value in values[start:stop]] for values in [v1, v2_A, v2_B]])
    ax.imshow(matrix, aspect="auto", interpolation="nearest", cmap=ListedColormap(list(colors.values())), vmin=0, vmax=3,
              extent=(0, (stop - start) / FPS, 2.5, -.5))
    local_contacts = contacts.loc[contacts.frame.between(start, stop - 1)]
    for side, local in local_contacts.groupby("team"):
        ax.scatter((local.frame - start) / FPS, np.full(len(local), 3), color=colors[side], marker="|", s=150)
    for name, local in features.loc[features.scene_id.eq(scene.scene_id)].groupby("video_event_type"):
        ax.scatter((local.frame - start) / FPS, np.full(len(local), 4), color=event_colors[name], marker="o", s=25 + 65 * local.video_event_score)
    unavailable = ~model_coverage[start:stop]
    for group in np.split(np.flatnonzero(unavailable), np.flatnonzero(np.diff(np.flatnonzero(unavailable)) > 1) + 1):
        if len(group):ax.axvspan(group[0] / FPS, (group[-1] + 1) / FPS, facecolor="none", hatch="//", edgecolor="black", alpha=.3)
    ax.set(title=scene.scene_id, xlim=(-.08, 12.08), ylim=(4.6, -.6), yticks=range(5),
           yticklabels=["v1", "v2-A", "v2-B", "Trackingkontakt", "T-DEED"], xlabel="Lokale Clipzeit [s]")
handles = [Patch(color=color, label=f"Modell/Tracking {label}") for label, color in colors.items()]
handles += [Line2D([0], [0], marker="o", linestyle="none", color=color, label=f"Video {name}") for name, color in event_colors.items()]
fig.legend(handles=handles, loc="outside upper center", ncol=4, fontsize=8)
fig.savefig(FIG / "01_before_after_timelines.png", dpi=150)
plt.show()

fig, axes = plt.subplots(5, 1, figsize=(13, 11), constrained_layout=True)
for ax, scene in zip(axes, clips.itertuples()):
    frames = np.arange(scene.first_frame, scene.stop_frame_exclusive)
    for name in event_colors:
        ax.plot((frames - scene.first_frame) / FPS, channels[name][frames], label=name, color=event_colors[name])
    ax.set(title=scene.scene_id, xlabel="Lokale Clipzeit [s]", ylabel="Positiver Support", ylim=(-.02, 1.02))
axes[0].legend(ncol=3, fontsize=8)
fig.savefig(FIG / "02_localized_video_support.png", dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(8, 3.5), constrained_layout=True)
ax.bar(comparison.model, comparison.changed_frames, color=["gray", "tab:blue", "tab:orange"])
for i, count in enumerate(comparison.changed_frames):ax.text(i, count, str(count), ha="center", va="bottom")
ax.set(title="Tatsächliche Labeländerungen gegenüber v1", ylabel="Frames", ylim=(0, max(1, comparison.changed_frames.max() * 1.1)))
ax.tick_params(axis="x", labelsize=8)
fig.savefig(FIG / "03_changed_frame_counts.png", dpi=150)
plt.show()

## Ergebnis

Beide Varianten ändern im bisherigen Pilot 0 Frames und 0 Besitzgrenzen. A bestätigt 15 Kontakte; B findet fünf zulässige Aktionsanker, aber keine wirksame Grenzverschiebung.

Das ist ein Nullergebnis. Die Pilotclips überdecken das öffentliche Fenster nicht; die früher berechneten identischen öffentlichen Scores von 0,564103 / 0,865360 können den Fusionsnutzen deshalb nicht bewerten. Die Event- und Teamqualität benötigt zulässige unabhängige Referenzdaten, bevor zusätzliche Modellkomplexität begründet ist.